# Flux footprints of the EC levels (Kljun et al. 2015, FFP)

Daily footprint climatologies of the 2, 16 and 26 m sonics for selected days, from the EddyPro output.
The FFP code lives outside this repo (`ffp_dir`).

## 0. Imports

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

src_path = str(Path.cwd().parent / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from ec.func_read_data import read_eddypro_data, read_data
from ec.func_footprint import ffp_inputs, footprint_climatology, plot_footprints, dem_local_grid

## 1. Configuration — the only cell you should need to edit

In [ ]:
"""CONFIGURATION — every other cell reads from the names defined here."""

DATA_ROOT = '/home/engbers/Documents/PhD'
ffp_dir   = '/home/engbers/Documents/Github/FFP_Python(2)'   # folder with calc_footprint_FFP_climatology.py
plot_dir  = '../plots'                                        # gitignored
ec_qc_level = 2                                               # read_eddypro_data masks fluxes with qc flag >= this

# one panel per day. 2024 is the single CSAT3 tower season (EddyPro sensor names *_DR, one folder level up;
# the only slow logger is SFC, in date folders under converted/).
# Wind direction comes from the wind vanes in the slow data, NOT from EddyPro: the fast data are double-rotated
# in Peddy before EddyPro, so EddyPro's wind_dir is only the orientation of the rotated axes (about 54 deg always).
# wd_vane = (slow logger, column) per level. The 16 m vane gave no data in 2025, and the 26 m vane differs from the
# 2 m vane by +2 deg in the season median, so 16 m uses the 26 m vane; in 2024 the tower levels use the 3 m SFC vane.
footprint_days = {
    '25 Apr 2025': dict(day='2025-04-25', folder=f'{DATA_ROOT}/EC_data_convert/2025/processed_HF',
                        filename_filter='v2_5min_biomet',
                        sensors={'2m': 'SFC', '16m': 'LOWER', '26m': 'UPPER'},
                        slow={'SFC': f'{DATA_ROOT}/EC_data_convert/2025/processed_slow/SFC',
                              'UPPER': f'{DATA_ROOT}/EC_data_convert/2025/processed_slow/UPPER'},
                        wd_vane={'2m': ('SFC', 'WD2'), '16m': ('UPPER', 'WD_26m'), '26m': ('UPPER', 'WD_26m')}),
    '13 May 2024': dict(day='2024-05-13', folder=f'{DATA_ROOT}/EC_data_convert',
                        filename_filter=None,
                        sensors={'2m': 'SFC_DR', '16m': 'CSAT_16m_DR', '26m': 'CSAT_26m_DR'},
                        slow={'SFC': f'{DATA_ROOT}/EC_data_convert/converted'},
                        wd_vane={'2m': ('SFC', 'WD2'), '16m': ('SFC', 'WD1'), '26m': ('SFC', 'WD1')}),
}

# FFP grid per level: [xmin, xmax, ymin, ymax] around the tower [m] and cell size [m]. The 2 m footprint is
# tens of metres long and needs a fine grid; the 16 and 26 m 80 % source areas reach several km in stable air.
ffp_grid = {
    '2m':  dict(domain=(-1500, 1500, -1500, 1500), dx=3),
    '16m': dict(domain=(-10000, 10000, -10000, 10000), dx=20),
    '26m': dict(domain=(-10000, 10000, -10000, 10000), dx=20),
}
ffp_rs     = (0.5, 0.8)                   # source-area contours drawn (solid = largest)
footprint_inset_pos = [[0.14, 0.05, 0.36, 0.36],   # ±250 m zoom per panel, [x0, y0, w, h] in axes fractions:
                       [0.57, 0.53, 0.40, 0.40]]   # bottom left in the first panel, top right in the second

# terrain under the footprints: REMA 32 m mosaic (EPSG:3031, heights above the WGS84 ellipsoid)
dem_path        = f'{DATA_ROOT}/CRYOWRF_files/PEA_brandon.tif'
tower_latlon    = (-71.94962, 23.35002)   # median GPS position logged by the SFC station (rmclatitude/rmclongitude)
dem_half_width  = 4500                    # local DEM grid around the tower [m]
dem_interval    = 10                      # contour interval [m]
dem_label_every = 50                      # label every this many metres

## 2. Load data

In [ ]:
"""READ IN EDDYPRO DATA and the slow loggers with the wind vanes, for every day
(QC flags >= ec_qc_level are masked by read_eddypro_data)"""

ec_by_day, slow_by_day = {}, {}
for title, cfg in footprint_days.items():
    day = pd.Timestamp(cfg['day'])
    ec_by_day[title] = {lvl: read_eddypro_data(cfg['folder'], sensor, qc=False, qc_level=ec_qc_level,
                                               filename_filter=cfg['filename_filter'])
                        for lvl, sensor in cfg['sensors'].items()}
    slow_by_day[title] = {logger: read_data(path, 'slow', logger, str(day - pd.Timedelta('1D')),
                                            str(day + pd.Timedelta('2D')), plot_data=False)
                          for logger, path in cfg['slow'].items()}

# elevation model on the footprint grid (x east, y north of the tower)
dem = dem_local_grid(dem_path, *tower_latlon, half_width=dem_half_width)


## 3. Footprints

In [ ]:
"""FOOTPRINT CLIMATOLOGY per day and level: all valid intervals of the day (UTC). The boundary-layer
height comes from u* and L (Nieuwstadt 1981), zm from EddyPro's (z-d)/L x L, the wind direction from the vane."""

footprints = {}
for title, cfg in footprint_days.items():
    day = cfg['day']
    footprints[title] = {}
    for lvl, fl in ec_by_day[title].items():
        logger, col = cfg['wd_vane'][lvl]
        inp = ffp_inputs(fl, slow_by_day[title][logger][col], f'{day} 00:00', f'{day} 23:59')
        footprints[title][lvl] = (footprint_climatology(inp, ffp_dir, rs=ffp_rs, **ffp_grid[lvl]), inp)

In [ ]:
"""FOOTPRINT MAP: 50 % (dashed) and 80 % (solid) source areas per level, one panel per day"""

fig, axes, footprint_table = plot_footprints(footprints, rs_plot=ffp_rs, inset_pos=footprint_inset_pos,
                                             dem=dem, dem_interval=dem_interval, dem_label_every=dem_label_every)
os.makedirs(plot_dir, exist_ok=True)
plt.savefig(f"{plot_dir}/footprint_ffp_{'_'.join(c['day'] for c in footprint_days.values())}.png", dpi=300, bbox_inches='tight')
plt.show()
footprint_table.round(2)